In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder ,StandardScaler # Import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score

In [ ]:
test= pd.read_csv('/content/test.csv')
train= pd.read_csv('/content/train.csv')

In [ ]:
train

In [ ]:
test.head()

In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.columns

In [ ]:
train.isnull().sum()

In [ ]:
train.describe()

In [ ]:
train.duplicated().sum()

# **transform data**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report

In [ ]:
train_only = [c for c in train.columns if c not in test.columns and c.lower() != 'id']
if len(train_only) != 1:
    raise ValueError('Expected exactly one target column present only in training data; inspect the dataset schema.')
target = train_only[0]
if train[target].isna().any():
    raise ValueError('Training labels contain missing values.')

In [ ]:
features = [c for c in train.columns if c != target and c.lower() != 'id']
if set(features) != {c for c in test.columns if c.lower() != 'id'}:
    raise ValueError('Training and inference feature columns do not match.')

In [ ]:
x = train[features].copy()
y = train[target].copy()
x_test_external = test[features].copy()

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
cat = x_train.select_dtypes(include=['object', 'category', 'string', 'bool']).columns.tolist()
num = [c for c in features if c not in cat]

In [ ]:
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
    ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat),
])

In [ ]:
x_test_external.head()

In [ ]:
model = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
model.fit(x_train, y_train)

In [ ]:
validation_predictions = model.predict(x_val)
print(classification_report(y_val, validation_predictions, zero_division=0))

# **Model**

In [ ]:
train.head()

In [ ]:
print('Target:', target)
print('Features:', features)

In [ ]:
predictions = model.predict(x_test_external)
assert len(predictions) == len(test)
predictions[:10]

In [ ]:
prediction_table = pd.DataFrame({target: predictions})
if 'id' in test.columns:
    prediction_table.insert(0, 'id', test['id'].to_numpy())
prediction_table.head()